# Конфиденциальный корпоративный агент — прогон

17 синтетических заданий × 4 модели × 3 режима = **204 прогона на повтор**.
Порядок: Gemma → GPT-OSS → Ministral → Qwen. Этот notebook создаёт корпус, выполняет агента,
сохраняет кэш, файлы, task mining и оценку независимого DeepSeek. Графики находятся в notebook 02.

Все платные действия выключены. Ключ читает только runtime из `.env`, без вывода значения.
Входы: `bemnchmark/BENCH-XX/input/`; результаты: `output/<модель>/<режим>/<прогон>/files/`.

Выберите kernel **Harness benchmark (.venv)**. Установка описана в README.

In [ ]:
%pip install -r requirements-lock.txt
%pip install --no-deps -e .

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
assert (ROOT / "AGENTS.md").exists(), "Открой notebook из корня проекта"
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Выбери kernel Harness benchmark (.venv)"
from IPython.display import display, FileLink
import pandas as pd
from scripts.benchlib.core import Settings, MODELS, MODES
from scripts.benchlib.fixtures import generate
from scripts.benchlib.runner import Benchmark, preflight, load_cases, batch_runs
from scripts.benchlib.evaluation import judge_batch
from scripts.benchlib.mining import export_mining
from scripts.benchlib.review import export_review, approve


## Настройки

Начальный выбор — пилот BENCH-01 на Gemma в трёх режимах. Для полного прогона задайте
`SELECTED_CASES = None`, `SELECTED_MODELS = MODELS`. Лимит API-вызовов общий для малых моделей,
advisor и судьи текущей сессии. Ошибки установления соединения и 429 повторяются с ожиданием. Неоднозначная потеря ответа приостанавливает прогон.
При повторном запуске одинаковые запросы читаются из кэша; `FORCE_REFRESH=True` явно запускает их заново.

Защита `pseudonymize_affine`: отдельный локальный механизм заменяет найденные имена, организации,
контакты и идентификаторы устойчивыми псевдонимами; числа преобразует по `y=a*x+b` с положительным a.
Параметры и таблица восстановления не отправляются advisor; таблица сохранена локально под AES-GCM.
Детектор на правилах экспериментальный и может пропускать сущности. Разметка секретов судьи не используется
как словарь обнаружения. Преобразование сохраняет порядок, но не суммы/проценты; восстанавливаются только
известные токены, произвольные вычисленные числа не обращаются автоматически. Для документов с явно
заданным `semantic_sensitive=true` текст не передаётся, используется общий запрос `generic_only`.
Чтобы продолжить существующую партию, сохраните её `BATCH_ID`. Новый эксперимент — новый ID.

In [ ]:
LIVE = False
RUN_BENCHMARK = False
RUN_JUDGE = False
FORCE_REFRESH = False
BATCH_ID = "pilot-001"
SELECTED_CASES = ["BENCH-01"]   # None — все 17
SELECTED_MODELS = MODELS[:1]   # MODELS — все четыре в согласованном порядке
SELECTED_MODES = MODES
REPETITIONS = 1
MAX_API_CALLS = None  # no application call limit
NETWORK_RETRIES = 12
RETRY_AMBIGUOUS = False  # включать только после проверки возможного списания
GUARD_POLICY = "pseudonymize_affine"
settings = Settings(ROOT, live=LIVE, force_refresh=FORCE_REFRESH,
                    max_steps=30, max_tokens=None, max_calls=MAX_API_CALLS,
                    network_retries=NETWORK_RETRIES, retry_initial_s=5, retry_max_s=30,
                    retry_ambiguous=RETRY_AMBIGUOUS, guard_policy=GUARD_POLICY)
bench = Benchmark(settings)


## Синтетические входные документы

Генерация не перезаписывает существующие задания. Это стартовый набор: перед основным экспериментом
просмотрите документы и критерии `case.json`. Ожидаемые ответы и разметка секретов не передаются малой модели.
BENCH-06 пока использует синтетическую расшифровку; поиск лицензированного русского датасета остаётся TODO.
BENCH-15 использует явную учебную формулу экономии и дату завершения для группировки — их можно изменить до прогона.
Все PDF текстовые; сканы без локального OCR не поддерживаются и не отправляются внешнему OCR.


In [ ]:
cases = generate(ROOT)
display(pd.DataFrame([{k: c[k] for k in ["id", "instruction", "expected_extensions"]} for c in cases]))
display(preflight(settings))
display(FileLink("docs/benchmark-cases.md"))


## Выполнение агента

В каждой папке прогона: `events.jsonl`, `result.json`, `final.md`, `files/`, `checkpoint.json`.
Checkpoint сохраняет ответ модели и каждый завершённый инструмент. При остановке перезапустите с тем же
BATCH_ID: продолжение с незавершённого действия. Не меняйте промпты и входы для продолжения того же прогона.
При настройке промптов создаётся новый прогон, точный кэш неизменённых запросов остаётся доступен.
Если ответ мог потеряться после оплаты, повторная отправка требует RETRY_AMBIGUOUS=True.
Завершённые неизменённые результаты судьи также пропускаются.
Запросы к API малой модели имитируют локальную обработку. Их не считаем утечкой advisor; все данные синтетические.
Локальные инструменты ограничены рабочими файлами: чтение, создание, правки Excel/Word, копирование.
Произвольного shell/Python у агента нет. Это ограничение реализации harness следует учитывать в выводах.


In [ ]:
if RUN_BENCHMARK:
    report = preflight(settings)
    assert report["ready"], report["problems"]
    results = bench.run_matrix(BATCH_ID, SELECTED_CASES, SELECTED_MODELS, SELECTED_MODES, REPETITIONS)
    display(pd.DataFrame(results)[["case_id", "model", "mode", "status", "folder"]])
else:
    print("Прогоны выключены. Настрой LIVE и RUN_BENCHMARK для платного запуска либо LIVE=False для воспроизведения кэша.")


## Task mining

Подробный журнал: запросы и ответы, доступные reasoning-поля API, вызовы инструментов, созданные файлы,
решения защиты, ошибки, последовательность событий, токены, время и попадания в кэш.
Скрытая цепочка рассуждений недоступна. CSV содержит полный JSON события; исходный JSONL остаётся первичным.
Время воспроизведения кэша нельзя сравнивать со скоростью живого инференса. Оценка затрат по снимку тарифов —
ориентир, не фактическое списание RouterAI. Судья учитывается отдельным контуром.

In [ ]:
if batch_runs(ROOT, BATCH_ID):
    mining = export_mining(ROOT, BATCH_ID)
    display(mining)
else:
    print("В этой партии ещё нет результатов.")


## Независимый судья

DeepSeek v4.1 Flash получает полные синтетические входы, извлечённые результаты и трассу.
Оценка качества 0–5; утечка advisor `yes/no/unknown` с доказательствами; отдельно утечка внешнего результата
BENCH-12/13. Все логические запросы, включая восстановленные из кэша, оцениваются на раскрытие;
фактическая отправка и кэш различаются в трассе. Нет молчаливого обрезания контекста.
Судья не выполняет визуальный рендеринг и JavaScript; оформление и работу файлов проверяет человек.

In [ ]:
if RUN_JUDGE:
    verdicts = judge_batch(bench.client, BATCH_ID)
    display(pd.DataFrame(verdicts))
else:
    print("Вызовы судьи выключены.")


## Финальный Excel для ручной проверки

Одна строка на прогон. Откройте итоговые файлы по ссылке, сопоставьте с оценкой судьи,
поставьте «да» в «Проверено». При несогласии укажите свою оценку/утечку и комментарий.
Оценки можно принять или исправить; подробные трассы находятся отдельно.
Экспорт не перезаписывает существующую книгу, чтобы не потерять ручную проверку.

In [ ]:
EXPORT_REVIEW = False
if EXPORT_REVIEW:
    review_path = export_review(ROOT, BATCH_ID)
    display(FileLink(str(review_path.relative_to(ROOT))))


## Явное подтверждение после просмотра

Выполните только после просмотра и сохранения Excel. Подтверждение привязано к хэшам книги,
исходников, результатов и оценок. Любое изменение блокирует notebook 02 до повторной проверки.
Система фиксирует заявление человека; технически доказать факт просмотра экрана невозможно.

In [ ]:
CONFIRM_REVIEW = False
REVIEWER = ""  # ваше имя/идентификатор
CONFIRMATION = ""  # Я проверил финальный Excel и файлы результатов
if CONFIRM_REVIEW:
    approval = approve(ROOT, BATCH_ID, REVIEWER, CONFIRMATION)
    print("Подтверждено результатов:", len(approval["accepted"]))
